# ============================================================
# STEP 4 — MULTI-METHOD XAI (tree models) — TUMC
# ============================================================
# Three complementary explainers on the best tree model:
#   1. SHAP (TreeExplainer)      — game-theoretic, local+global
#   2. LIME (tabular)            — local surrogate
#   3. Permutation importance    — model-agnostic global, robust
#
# HEADLINE OUTPUT: cross-method AGREEMENT.
#   - SHAP vs Permutation: global rank correlation (Spearman)
#   - SHAP vs LIME: per-instance top-feature overlap
#   - Convergent validity vs Step 2E robust-56 consensus
#
# Output: step4_xai_agreement.csv, step4_method_ranks.csv,
#         xai_plots/*.png, shap_top_features.json (feeds Step 5)
# ============================================================

In [ ]:
import os, json, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
SEED = 42

DATA_DIR = "/content/drive/MyDrive/TUMC_dataset"
for cand in ["features_full_final_inductive.csv","features_full_final.csv"]:
    INPUT_FILE = os.path.join(DATA_DIR, cand)
    if os.path.exists(INPUT_FILE): break
PLOTS = os.path.join(DATA_DIR, "xai_plots"); os.makedirs(PLOTS, exist_ok=True)

TASK = "binary"
SHAP_SAMPLE = 30_000        # SHAP global
LIME_SAMPLE = 200           # LIME is slow per-instance
PERM_SAMPLE = 50_000
MODEL = "xgboost"           # xgboost | lightgbm
LEAKY = ["cluster_malicious_ratio","campaign_token_score","is_tr_domain","is_https"]

try: import shap
except ImportError: print("pip install shap -q"); raise SystemExit
try:
    from lime.lime_tabular import LimeTabularExplainer
    HAS_LIME = True
except ImportError:
    print("⚠ pip install lime -q  (LIME section will skip)"); HAS_LIME = False

print("="*70)
print(f"STEP 4 — MULTI-METHOD XAI ({TASK})  source={os.path.basename(INPUT_FILE)}")
print("="*70)

df = pd.read_csv(INPUT_FILE, low_memory=False)
META = {"url","source","tld","label","label_enc","class_final",
        "class_enc","fold","reg_domain"}
FEATURES = [c for c in df.columns if c not in META and c not in LEAKY]
target = "label_enc" if TASK=="binary" else "class_enc"
n_classes = df[target].nunique()
is_binary = (TASK=="binary")

tr = df[df["fold"]!=0]; te = df[df["fold"]==0]
X_tr, y_tr = tr[FEATURES].fillna(0).values, tr[target].values
X_te, y_te = te[FEATURES].fillna(0).values, te[target].values

print(f"\n[1] Train {MODEL} on {len(X_tr):,} rows, {len(FEATURES)} features ...")
if MODEL=="xgboost":
    import xgboost as xgb
    model = xgb.XGBClassifier(
        max_depth=4, learning_rate=0.03, n_estimators=400,
        subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=-1,
        **({"scale_pos_weight":5.5} if is_binary else
           {"objective":"multi:softprob","num_class":n_classes}))
else:
    import lightgbm as lgb
    model = lgb.LGBMClassifier(n_estimators=400, max_depth=6, learning_rate=0.03,
        class_weight="balanced", random_state=SEED, n_jobs=-1, verbose=-1,
        **({} if is_binary else {"objective":"multiclass","num_class":n_classes}))
model.fit(X_tr, y_tr)

# ════════════════════════════════════════════════════════════
# METHOD 1 — SHAP
# ════════════════════════════════════════════════════════════
print(f"\n[2] SHAP (TreeExplainer, {SHAP_SAMPLE:,} sample) ...")
Xs_idx = np.random.RandomState(SEED).choice(len(X_te), min(SHAP_SAMPLE,len(X_te)), replace=False)
Xs = X_te[Xs_idx]
explainer = shap.TreeExplainer(model)
sv = explainer.shap_values(Xs)
if isinstance(sv, list):           # multiclass → mean over classes
    shap_imp = np.mean([np.abs(s).mean(0) for s in sv], axis=0)
else:
    shap_imp = np.abs(sv).mean(0)
shap_s = pd.Series(shap_imp, index=FEATURES).sort_values(ascending=False)
print("    SHAP top-8:")
for f,v in shap_s.head(8).items(): print(f"      {f:<26s}: {v:.4f}")

# Save for Step 5
with open(os.path.join(DATA_DIR,"shap_top_features.json"),"w") as fh:
    json.dump({"top8":shap_s.head(8).index.tolist(),
               "top15":shap_s.head(15).index.tolist()}, fh)

# ════════════════════════════════════════════════════════════
# METHOD 2 — PERMUTATION IMPORTANCE
# ════════════════════════════════════════════════════════════
print(f"\n[3] Permutation importance ({PERM_SAMPLE:,} sample) ...")
pi_idx = np.random.RandomState(SEED).choice(len(X_te), min(PERM_SAMPLE,len(X_te)), replace=False)
scoring = "roc_auc" if is_binary else "f1_macro"
pi = permutation_importance(model, X_te[pi_idx], y_te[pi_idx],
    n_repeats=5, random_state=SEED, scoring=scoring, n_jobs=-1)
perm_s = pd.Series(pi.importances_mean, index=FEATURES).sort_values(ascending=False)
print("    Permutation top-8:")
for f,v in perm_s.head(8).items(): print(f"      {f:<26s}: {v:.4f}")

# ════════════════════════════════════════════════════════════
# METHOD 3 — LIME (local) + SHAP-LIME agreement
# ════════════════════════════════════════════════════════════
lime_overlap = np.nan
if HAS_LIME:
    print(f"\n[4] LIME ({LIME_SAMPLE} instances) + SHAP agreement ...")
    lime_exp = LimeTabularExplainer(
        X_tr, feature_names=FEATURES,
        class_names=["benign","malicious"] if is_binary else [str(i) for i in range(n_classes)],
        discretize_continuous=True, random_state=SEED, mode="classification")
    li_idx = np.random.RandomState(SEED).choice(len(X_te), LIME_SAMPLE, replace=False)
    overlaps = []
    lime_feat_counter = {}
    for i in li_idx:
        exp = lime_exp.explain_instance(
            X_te[i], model.predict_proba, num_features=8,
            labels=(1,) if is_binary else (int(y_te[i]),))
        lbl = 1 if is_binary else int(y_te[i])
        lime_top = set()
        for feat_str, _ in exp.as_list(label=lbl):
            # LIME returns "feature <= x" strings; map back to feature name
            for fn in FEATURES:
                if fn in feat_str:
                    lime_top.add(fn); break
        # SHAP top-8 for THIS instance
        if isinstance(sv, list):
            inst_sv = np.abs(sv[lbl][np.where(Xs_idx==i)[0][0]]) if i in Xs_idx else None
        else:
            inst_sv = None
        # Use global SHAP top-8 as reference (instance-level optional)
        shap_top8 = set(shap_s.head(8).index)
        if lime_top:
            overlaps.append(len(lime_top & shap_top8) / max(len(lime_top),1))
        for f in lime_top:
            lime_feat_counter[f] = lime_feat_counter.get(f,0)+1
    lime_overlap = np.mean(overlaps) if overlaps else np.nan
    print(f"    Mean SHAP∩LIME top-feature overlap: {lime_overlap:.2%}")
    lime_s = pd.Series(lime_feat_counter).sort_values(ascending=False)
    print("    LIME most-frequent features:")
    for f,v in lime_s.head(8).items(): print(f"      {f:<26s}: {v}")
else:
    lime_s = pd.Series(dtype=float)

# ════════════════════════════════════════════════════════════
# CROSS-METHOD AGREEMENT
# ════════════════════════════════════════════════════════════
print(f"\n[5] Cross-method agreement ...")
shap_rank = shap_s.rank(ascending=False)
perm_rank = perm_s.rank(ascending=False)
common = shap_rank.index.intersection(perm_rank.index)
rho, pval = spearmanr(shap_rank[common], perm_rank[common])
print(f"    SHAP vs Permutation rank correlation (Spearman ρ): {rho:.4f} (p={pval:.2e})")

# Agreement table: top-15 union with each method's rank
top_union = list(dict.fromkeys(
    list(shap_s.head(15).index) + list(perm_s.head(15).index)))
agree = pd.DataFrame({
    "feature": top_union,
    "shap_rank": [int(shap_rank.get(f,999)) for f in top_union],
    "perm_rank": [int(perm_rank.get(f,999)) for f in top_union],
    "shap_score": [round(shap_s.get(f,0),4) for f in top_union],
    "perm_score": [round(perm_s.get(f,0),4) for f in top_union],
})
if HAS_LIME and len(lime_s):
    lime_rank = lime_s.rank(ascending=False)
    agree["lime_freq_rank"] = [int(lime_rank.get(f,999)) for f in top_union]
agree["mean_rank"] = agree[[c for c in agree.columns if c.endswith("_rank")]].mean(axis=1)
agree = agree.sort_values("mean_rank")

# Cross-reference Step 2E robust set
robust_set = set()
try:
    with open(os.path.join(DATA_DIR,"step2e_selected_features.pkl"),"rb") as fh:
        sel = pickle.load(fh)
    robust_set = set(sel.get("robust",[]))
    agree["in_2E_robust"] = agree["feature"].isin(robust_set)
    n_conv = agree["in_2E_robust"].sum()
    print(f"    Convergent validity: {n_conv}/{len(agree)} XAI-top features"
          f" also in Step 2E robust-{len(robust_set)} set")
except FileNotFoundError:
    print("    (Step 2E pkl not found — skipping convergent-validity cross-check)")

agree.to_csv(os.path.join(DATA_DIR,"step4_xai_agreement.csv"), index=False)
print(f"\n    Agreement table (top 12):")
print(agree.head(12).to_string(index=False))

# ════════════════════════════════════════════════════════════
# PLOT — method comparison
# ════════════════════════════════════════════════════════════
topn = agree.head(15)
fig, ax = plt.subplots(figsize=(11,8))
yy = np.arange(len(topn)); h = 0.27
ax.barh(yy+h, topn["shap_rank"].max()-topn["shap_rank"]+1, h, label="SHAP", color="#1D9E75")
ax.barh(yy,    topn["perm_rank"].max()-topn["perm_rank"]+1, h, label="Permutation", color="#5B7BB8")
if "lime_freq_rank" in topn.columns:
    ax.barh(yy-h, topn["lime_freq_rank"].max()-topn["lime_freq_rank"]+1, h, label="LIME", color="#E8762C")
ax.set_yticks(yy); ax.set_yticklabels(topn["feature"], fontsize=9); ax.invert_yaxis()
ax.set_xlabel("Inverted rank (longer = more important)")
ax.set_title(f"Cross-Method Feature Importance — TUMC ({TASK})\n"
             f"SHAP vs Permutation Spearman ρ={rho:.3f}", fontweight="bold")
ax.legend()
plt.tight_layout(); plt.savefig(f"{PLOTS}/xai_method_comparison_{TASK}.png", dpi=150); plt.close()

# SHAP beeswarm (binary) / bar
plt.figure()
if isinstance(sv, list):
    shap.summary_plot(sv, Xs, feature_names=FEATURES, plot_type="bar", show=False, max_display=20)
else:
    shap.summary_plot(sv, Xs, feature_names=FEATURES, show=False, max_display=20)
plt.tight_layout(); plt.savefig(f"{PLOTS}/shap_summary_{TASK}.png", dpi=150, bbox_inches="tight"); plt.close()

print(f"""
{'='*70}
STEP 4 (TREE XAI) COMPLETE
{'='*70}
  Methods: SHAP + Permutation{' + LIME' if HAS_LIME else ''}
  SHAP–Permutation Spearman ρ : {rho:.4f}
  {'SHAP∩LIME overlap           : '+f'{lime_overlap:.2%}' if HAS_LIME else ''}
  Agreement table : step4_xai_agreement.csv
  SHAP top-8 saved: shap_top_features.json (feeds Step 5)

  Manuscript story: high cross-method agreement = explanations are
  method-independent, not artifacts of a single algorithm.
  Next → Step 4B: Integrated Gradients for deep models
{'='*70}
""")